# Supervised Learning – Regression mit einer Stufenfunktion

**Ziel:** Sie lernen, wie eine **Stufenfunktion (Step Function)** eine numerische Eingabevariable in Intervalle einteilt und für jedes Intervall einen konstanten Vorhersagewert lernt.

**Ablauf:** 1. CSV hochladen · 2. Feature und Zielvariable wählen · 3. Intervalle definieren · 4. Trainings-/Testdaten aufteilen · 5. Stufenfunktion trainieren · 6. Visualisieren · 7. Modellgüte bewerten · 8. Mit linearer Regression vergleichen.

**Voreinstellung:** Advertising-Datensatz von James et al.: `TV` → `sales`. Die Daten liegen in **Tausend US-Dollar** (Werbeausgaben) und **Tausend Stück** (Absatz) vor. Andere CSV-Dateien mit numerischen Variablen können ebenso verwendet werden.

*Vorkenntnisse: keine Programmierung erforderlich. Ändern Sie nur die Werte im Abschnitt „Einstellungen“.*

## 1. CSV-Datei hochladen

Führen Sie die Zelle aus und wählen Sie eine CSV-Datei von Ihrem Computer aus (z. B. `Advertising.csv`).

In [ ]:
from google.colab import files
uploaded = files.upload()

## 2. Daten einlesen und prüfen

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed:')]
print('Datensatz:', filename, '| Zeilen:', len(df))
print('Spalten:', list(df.columns))
display(df.head())

## 3. Einstellungen: Feature, Zielvariable und Intervalle

Ändern Sie **nur die Werte rechts vom Gleichheitszeichen**. Die Intervallgrenzen haben dieselbe Einheit wie das ausgewählte Feature.

**Beispiel Advertising:** `TV=75` steht für 75.000 US-Dollar. Grenzen 75, 150, 225 ergeben vier Intervalle. Alternativ können Grenzen automatisch über **Quantile der Trainingsdaten** berechnet werden.

**Wichtig:** Es wird **ein numerisches Feature** untersucht; zusätzliche Features werden in diesem anschaulichen Beispiel bewusst nicht berücksichtigt.

In [ ]:
TARGET = 'sales'             # numerische Zielvariable, z. B. sales
FEATURE = 'TV'               # numerisches Feature, z. B. TV oder Age
METHOD = 'manual'            # 'manual' oder 'quantile'
CUTS = [75, 150, 225]        # nur bei METHOD='manual'; strikt aufsteigend
N_INTERVALS = 4              # nur bei METHOD='quantile'
TEST_SIZE = 0.20
RANDOM_STATE = 42

assert TARGET in df.columns and FEATURE in df.columns, 'Feature oder Zielvariable nicht vorhanden.'
assert FEATURE != TARGET, 'Feature und Zielvariable müssen verschieden sein.'
assert METHOD in ('manual', 'quantile')
assert df[FEATURE].dtype.kind in 'iuf' and df[TARGET].dtype.kind in 'iuf', 'Wählen Sie numerische Spalten.'
if METHOD == 'manual':
    assert all(np.isfinite(CUTS)) and all(a < b for a, b in zip(CUTS, CUTS[1:])), 'CUTS muss streng aufsteigend sein.'
else:
    assert isinstance(N_INTERVALS, int) and N_INTERVALS >= 2

data = df[[FEATURE, TARGET]].replace([np.inf, -np.inf], np.nan).dropna().copy()
assert len(data) >= 10, 'Zu wenig vollständige Beobachtungen.'
print('Verwendete Beobachtungen:', len(data))
print(f'{FEATURE}: Minimum = {data[FEATURE].min():.2f}; Maximum = {data[FEATURE].max():.2f}')

## 4. Trainings- und Testdaten bilden

Nur Trainingsdaten werden verwendet, um die konstanten Werte der Stufenfunktion zu schätzen. Bei der Quantil-Option werden auch die Intervallgrenzen ausschließlich aus Trainingsdaten ermittelt.

In [ ]:
train, test = train_test_split(data, test_size=TEST_SIZE, random_state=RANDOM_STATE)
if METHOD == 'manual':
    cuts = list(map(float, CUTS))
else:
    cuts = np.unique(np.quantile(train[FEATURE], np.linspace(0, 1, N_INTERVALS + 1)[1:-1])).tolist()
    assert cuts, 'Zu wenige unterschiedliche Feature-Werte für Quantilintervalle.'

bins = [-np.inf] + cuts + [np.inf]
labels = [f'Intervall {i+1}' for i in range(len(bins)-1)]
train = train.copy()
test = test.copy()
train['Intervall'] = pd.cut(train[FEATURE], bins=bins, labels=labels, right=False)
test['Intervall'] = pd.cut(test[FEATURE], bins=bins, labels=labels, right=False)

print('Trainingsdaten:', len(train), '| Testdaten:', len(test))
print('Grenzen:', cuts)
print('Intervalle: [untere Grenze, obere Grenze), z. B. [75, 150)')

## 5. Stufenfunktion trainieren

Für jedes Intervall berechnen wir den **durchschnittlichen Zielwert aus den Trainingsdaten**. Dieser Mittelwert minimiert die Summe der quadratischen Fehler innerhalb des jeweiligen Intervalls.

Falls ein vorgegebenes Intervall keine Trainingsbeobachtung enthält, verwenden wir hier als transparente Ersatzregel den **globalen Trainingsmittelwert**. Solche Intervalle werden ausgewiesen.

In [ ]:
train_mean = train[TARGET].mean()
stats = train.groupby('Intervall', observed=False)[TARGET].agg(['size','mean','std'])
stats = stats.reindex(labels)
empty = stats['size'].fillna(0).eq(0)
if empty.any():
    print('Hinweis: Leere Trainingsintervalle:', list(stats.index[empty]))
stats['Vorhersage'] = stats['mean'].fillna(train_mean)

means = stats['Vorhersage'].to_dict()
train['Prognose_Stufe'] = train['Intervall'].astype(str).map(means).astype(float)
test['Prognose_Stufe'] = test['Intervall'].astype(str).map(means).astype(float)
assert test['Prognose_Stufe'].notna().all()

summary = pd.DataFrame({
    'Bereich': [f"{'−∞' if i == 0 else f'{bins[i]:g}'} bis {'+∞' if i == len(labels)-1 else f'{bins[i+1]:g}'}" for i in range(len(labels))],
    'Beobachtungen im Training': stats['size'].astype(int).values,
    f'geschätzter {TARGET}-Wert': stats['Vorhersage'].values
}, index=labels)
display(summary.round(3))

## 6. Stufenfunktion und Beobachtungen visualisieren

**Orange Punkte:** Trainingsbeobachtungen. **Dunkle Stufenlinie:** gelernte Vorhersage. Die gestrichelten Linien kennzeichnen die Intervallgrenzen.

In [ ]:
xx = np.linspace(data[FEATURE].min(), data[FEATURE].max(), 900)
xx_bins = pd.cut(pd.Series(xx), bins=bins, labels=labels, right=False).astype(str)
yy = xx_bins.map(means).to_numpy(dtype=float)

plt.figure(figsize=(10, 5.5))
plt.scatter(train[FEATURE], train[TARGET], alpha=.50, s=22, label='Training: Beobachtungen', color='darkorange')
plt.plot(xx, yy, linewidth=2.7, label='Trainierte Stufenfunktion', color='midnightblue')
for c in cuts:
    if data[FEATURE].min() <= c <= data[FEATURE].max():
        plt.axvline(c, linestyle='--', color='gray', alpha=.7)
plt.xlabel(FEATURE)
plt.ylabel(TARGET)
plt.title(f'Stufenfunktion: {FEATURE} → {TARGET}')
plt.legend()
plt.grid(alpha=.15)
plt.show()

## 7. Modellgüte auf Testdaten bewerten

- **SSE / RSS:** Summe der quadrierten Vorhersagefehler
- **MSE:** mittlerer quadrierter Vorhersagefehler
- **RMSE:** Quadratwurzel aus MSE (in der Einheit der Zielvariable)
- **MAE:** mittlerer absoluter Vorhersagefehler
- **R²:** Vorhersagegüte relativ zum Mittelwert der Test-Zielvariable (auf Testdaten auch negativ möglich)

Niedrigere Fehlerwerte und ein höheres R² sind bei **derselben Testmenge** besser.

In [ ]:
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        'R²': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'SSE / RSS': mse * len(y_true)
    }

print('Stufenfunktion – Testdaten:')
display(pd.Series(metrics(test[TARGET], test['Prognose_Stufe']), name='Wert').round(3).to_frame())
display(test[[FEATURE,TARGET,'Intervall','Prognose_Stufe']].head(10))

## 8. Vergleich: lineare Regression (dasselbe Feature, dieselben Daten)

Zum fairen Vergleich verwenden wir den identischen Trainings-/Test-Split und dieselbe Eingabevariable. Das ist ein erster Modellvergleich, **noch keine endgültige Modellauswahl**.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(train[[FEATURE]], train[TARGET])
test['Prognose_Linear'] = linear_model.predict(test[[FEATURE]])

comparison = pd.DataFrame({
    'Stufenfunktion': metrics(test[TARGET], test['Prognose_Stufe']),
    'Linear': metrics(test[TARGET], test['Prognose_Linear'])
}).T
print(f'Lineares Modell: {TARGET} = {linear_model.intercept_:.3f} + {linear_model.coef_[0]:.3f} × {FEATURE}')
display(comparison.round(3))

plt.figure(figsize=(10, 5.5))
plt.scatter(test[FEATURE], test[TARGET], alpha=.5, s=24, color='darkorange', label='Testdaten')
plt.plot(xx, yy, linewidth=2.5, color='midnightblue', label='Stufenfunktion')
plt.plot(xx, linear_model.predict(pd.DataFrame({FEATURE: xx})), linewidth=2, color='forestgreen', label='Lineare Regression')
plt.xlabel(FEATURE);plt.ylabel(TARGET)
plt.title('Vergleich der Hypothesenräume')
plt.legend();plt.grid(alpha=.15);plt.show()

## 9. Reflexionsaufgaben

1. **Hypothesenraum:** Welche Funktionen sind zugelassen, nachdem Sie die Intervallgrenzen festgelegt haben?
2. **Lernen:** Woher stammen die Vorhersagewerte innerhalb der Intervalle?
3. **Grenzen ändern:** Probieren Sie zwei statt vier Intervalle. Wie verändert sich die Stufenfunktion?
4. **Testfehler vergleichen:** Ist der Test-RMSE für die Stufenfunktion oder die lineare Regression niedriger?
5. **Grenze:** Warum können zwei deutlich unterschiedliche Feature-Werte dieselbe Vorhersage bekommen?
6. **Überanpassung:** Warum können sehr viele schmale Intervalle problematisch sein?

### Methodischer Hinweis

Die Intervallgrenzen sind Teil der **Modellspezifikation**. Die Mittelwerte innerhalb der Intervalle sind die **gelernten Modellparameter**. Für eine belastbare Auswahl der Grenzen sollten diese ausschließlich auf Trainingsdaten (z. B. mittels Cross-Validation) abgestimmt werden. **Testdaten dürfen nicht zur Optimierung der Intervallgrenzen verwendet werden.**

Dieses Notebook modelliert absichtlich jeweils **ein** numerisches Feature. Für eine multiple Regression mit mehreren Features wäre eine erweiterte Modellierung erforderlich.